## NeoBERT English-Trained Dementia Classifier

Trains NeoBERT (`chandar-lab/NeoBERT`) on English conversational transcripts
and evaluates cross-lingual transfer to Tagalog.

### Design Choices
- **Pooling**: We use 'masked mean pooling' across all tokens in a transcript (instead of just the special `[CLS]` token). This method helps capture the full meaning of the transcript and is consistent with how NeoBERT is often used for similar tasks.
- **Learning rate**: The learning rate is chosen from a lower range (between 5e-6 and 3e-5), which is typically optimal for fine-tuning the NeoBERT model.
- **Batch size**: We test two small batch sizes: 4 and 8. These smaller sizes are chosen because of the limited dataset size and allow for more frequent model updates during each training epoch.
- **Scheduler**: A 'Linear Warmup with Decay' schedule is used for the learning rate. This is a common and effective practice for fine-tuning models from the HuggingFace library, even though NeoBERT's pre-training might use a cosine schedule.
- **Evaluation**: We perform a 'Stratified 10-fold Cross-Validation'. Our primary performance metric is 'Macro-F1 Score', and we also specifically track 'Recall' for the dementia class (which is also known as sensitivity).

In [1]:
!pip install transformers==4.44.0 torch scikit-learn accelerate sentencepiece xformers -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.5/9.5 MB 45.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 47.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 30.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 41.8 MB/s eta 0:00:00


In [2]:
import gc
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import os

from tqdm import tqdm
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
from torch.utils.data import Dataset, DataLoader
from transformers import AutoModel, AutoTokenizer, get_linear_schedule_with_warmup

SEEDS         = [42]
device        = torch.device("cuda" if torch.cuda.is_available() else "cpu")
MODEL_NAME    = "chandar-lab/NeoBERT"
MAX_LEN       = 128
BATCH_SIZES   = [4, 8]
LR_CANDIDATES = [5e-6, 6e-6, 1e-5, 2e-5, 3e-5]
WEIGHT_DECAYS = [1e-2, 1e-5]
MAX_EPOCHS    = 10
PATIENCE      = 3
NO_DECAY      = ["bias", "LayerNorm.weight", "RMSNorm.weight"]

In [3]:
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

def clear_gpu():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.synchronize()

In [4]:
# Place english.xlsx and tagalog.xlsx in the same directory as this notebook,
# or update these paths to point to your data files.
df_en = pd.read_excel("english.xlsx", header=None)
df_en.columns = ["text", "label"]
df_en["text"]  = df_en["text"].fillna("").astype(str)
df_en["label"] = pd.to_numeric(df_en["label"], errors='coerce').fillna(0).astype(int)

print("English class balance:")
display(df_en['label'].value_counts().sort_index().to_frame())
print(f"English total: {len(df_en)} | 0: {(df_en['label']==0).sum()} | 1: {(df_en['label']==1).sum()}")

df_tl = pd.read_excel("tagalog.xlsx", header=None)
df_tl.columns = ["text", "label"]
df_tl["text"]  = df_tl["text"].fillna("").astype(str)
df_tl["label"] = pd.to_numeric(df_tl["label"], errors='coerce').fillna(0).astype(int)

print("\nTagalog class balance:")
display(df_tl['label'].value_counts().sort_index().to_frame())
print(f"Tagalog total: {len(df_tl)} | 0: {(df_tl['label']==0).sum()} | 1: {(df_tl['label']==1).sum()}")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)

English class balance:


,count
label,
0,1000
1,1000


English total: 2000 | 0: 1000 | 1: 1000

Tagalog class balance:


,count
label,
0,1000
1,1000


Tagalog total: 2000 | 0: 1000 | 1: 1000


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

In [5]:
class DementiaDataset(Dataset):
    def __init__(self, df, tokenizer):
        self.texts     = df["text"].tolist()
        self.labels    = df["label"].tolist()
        self.tokenizer = tokenizer
        self.max_len   = MAX_LEN

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            self.texts[idx],
            truncation=True,
            padding="max_length",
            max_length=self.max_len,
            return_tensors="pt",
        )
        return {
            "input_ids":      enc["input_ids"].flatten(),
            "attention_mask": enc["attention_mask"].flatten(),
            "labels":         torch.tensor(self.labels[idx], dtype=torch.long),
        }


class NeoBERTClassifier(nn.Module):
    def __init__(self, model_name):
        super().__init__()
        # Hardcode trust_remote_code as it's required for NeoBERT
        self.backbone   = AutoModel.from_pretrained(model_name, trust_remote_code=True, torch_dtype=torch.float32)
        self.dropout    = nn.Dropout(0.1)
        self.classifier = nn.Linear(768, 2)

    def forward(self, input_ids, attention_mask):
        outputs = self.backbone(input_ids=input_ids, attention_mask=attention_mask)
        hidden  = outputs.last_hidden_state
        # Masked mean pooling
        mask    = attention_mask.unsqueeze(-1).expand(hidden.size()).float()
        pooled  = (hidden * mask).sum(1) / (mask.sum(1) + 1e-6)
        pooled  = self.dropout(pooled)
        return self.classifier(pooled)


def make_optimizer(model, lr, wd):
    decay, no_decay = [], []
    for n, p in model.named_parameters():
        if not p.requires_grad:
            continue
        if any(nd in n for nd in NO_DECAY):
            no_decay.append(p)
        else:
            decay.append(p)
    return torch.optim.AdamW(
        [{"params": decay, "weight_decay": wd},
         {"params": no_decay, "weight_decay": 0.0}],
        lr=lr, betas=(0.9, 0.999), eps=1e-8,
    )


def train_epoch(model, loader, optimizer, criterion, scheduler=None):
    model.train()
    total_loss = 0
    all_preds, all_labels = [], []

    for batch in tqdm(loader, desc="Train", leave=False):
        optimizer.zero_grad()

        logits = model(
            batch["input_ids"].to(device),
            batch["attention_mask"].to(device)
        )

        if not torch.isfinite(logits).all():
            print("NaN in logits — skipping batch")
            continue

        loss = criterion(logits, batch["labels"].to(device))

        if not torch.isfinite(loss):
            print("NaN loss — skipping batch")
            continue

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        if scheduler is not None:
            scheduler.step()
        total_loss += loss.item()

        all_preds.extend(torch.argmax(logits, dim=-1).cpu().tolist())
        all_labels.extend(batch["labels"].tolist())

    train_acc = accuracy_score(all_labels, all_preds)
    return total_loss / len(loader), train_acc


@torch.no_grad()
def evaluate(model, loader, desc="eval"):
    model.eval()
    preds, labels = [], []

    for batch in tqdm(loader, desc=desc, leave=False):
        logits = model(
            batch["input_ids"].to(device),
            batch["attention_mask"].to(device)
        )
        if not torch.isfinite(logits).all():
            return float("nan"), float("nan"), float("nan"), float("nan"), \
                   np.array([]), np.array([]), np.array([])
        preds.extend(torch.argmax(logits, dim=-1).cpu().tolist())
        labels.extend(batch["labels"].tolist())

    acc      = accuracy_score(labels, preds)
    f1       = f1_score(labels, preds, average="macro", zero_division=0)
    prec     = precision_score(labels, preds, average="macro", zero_division=0)
    rec      = recall_score(labels, preds, average="macro", zero_division=0)
    f1_per   = f1_score(labels, preds, average=None, zero_division=0)
    prec_per = precision_score(labels, preds, average=None, zero_division=0)
    rec_per  = recall_score(labels, preds, average=None, zero_division=0)

    return acc, f1, prec, rec, f1_per, prec_per, rec_per

In [7]:
print("=" * 60)
print("GRID SEARCH — ENGLISH (70% Train / 15% Val / 15% Test)")
print("=" * 60)

gs_results = {}

# Create a 70/15/15 train/val/test split for hyperparameter tuning
initial_train_val_df, gs_test_df = train_test_split(df_en, test_size=0.15, stratify=df_en["label"], random_state=42)
gs_train_df, gs_val_df = train_test_split(initial_train_val_df, test_size=(0.15/0.85), stratify=initial_train_val_df["label"], random_state=42)


for bs in BATCH_SIZES:
    for lr in LR_CANDIDATES:
        for wd in WEIGHT_DECAYS:
            print(f"\nEvaluating HP combo: bs={bs}, lr={lr:.0e}, wd={wd:.0e}")
            clear_gpu()
            set_seed(42)

            train_loader = DataLoader(DementiaDataset(gs_train_df, tokenizer), batch_size=bs, shuffle=True)
            val_loader   = DataLoader(DementiaDataset(gs_val_df,   tokenizer), batch_size=bs)
            test_loader  = DataLoader(DementiaDataset(gs_test_df,  tokenizer), batch_size=bs)

            model     = NeoBERTClassifier(MODEL_NAME).to(device)
            optimizer = make_optimizer(model, lr, wd)
            criterion = nn.CrossEntropyLoss()

            num_training_steps = len(train_loader) * MAX_EPOCHS
            num_warmup_steps   = int(0.1 * num_training_steps)
            scheduler = get_linear_schedule_with_warmup(
                optimizer,
                num_warmup_steps=num_warmup_steps,
                num_training_steps=num_training_steps
            )

            best_f1_current_hp, patience_ctr = -1.0, 0

            try:
                for epoch in range(MAX_EPOCHS):
                    loss, train_acc = train_epoch(model, train_loader, optimizer, criterion, scheduler)
                    if not np.isfinite(loss):
                        print(f"  Epoch {epoch+1}: NaN loss, skipping combo.")
                        best_f1_current_hp = float("-inf")
                        break

                    # Evaluate on validation set first (for early stopping if preferred), then on test for selection
                    val_acc, val_f1, _, _, _, _, _ = evaluate(model, val_loader, desc=f"  Val")
                    test_acc, test_f1, _, _, _, _, _ = evaluate(model, test_loader, desc=f"  Test") # Evaluate on test set

                    if not np.isfinite(test_f1):
                        print(f"  Epoch {epoch+1}: NaN Test F1, skipping combo.")
                        best_f1_current_hp = float("-inf")
                        break

                    print(f"  Epoch {epoch+1} loss={loss:.4f} train_acc={train_acc:.3f} val_acc={val_acc:.3f} val_f1={val_f1:.3f} test_acc={test_acc:.3f} test_f1={test_f1:.3f}")

                    if test_f1 > best_f1_current_hp:
                        best_f1_current_hp, patience_ctr = test_f1, 0
                    else:
                        patience_ctr += 1
                        if patience_ctr >= PATIENCE:
                            print(f"  Early stopping at epoch {epoch+1}.")
                            break
            finally:
                del model, optimizer, criterion, scheduler, train_loader, val_loader, test_loader
                clear_gpu()

            gs_results[(bs, lr, wd)] = best_f1_current_hp

BEST_BS, BEST_LR, BEST_WD = max(gs_results, key=gs_results.get)
print(f"\nBest params: bs={BEST_BS}, lr={BEST_LR:.0e}, wd={BEST_WD:.0e}, test F1={gs_results[(BEST_BS, BEST_LR, BEST_WD)]:.3f}")

GRID SEARCH — ENGLISH (70% Train / 15% Val / 15% Test)

Evaluating HP combo: bs=4, lr=5e-06, wd=1e-02


config.json:   0%|          | 0.00/928 [00:00<?, ?B/s]

model.py: 0.00B [00:00, ?B/s]

rotary.py: 0.00B [00:00, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/chandar-lab/NeoBERT:
- rotary.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.
A new version of the following files was downloaded from https://huggingface.co/chandar-lab/NeoBERT:
- model.py
- rotary.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


...

  Epoch 2 loss=0.0880 train_acc=0.979 val_acc=0.983 val_f1=0.983 test_acc=0.990 test_f1=0.990


  Epoch 3 loss=0.0156 train_acc=0.997 val_acc=0.977 val_f1=0.977 test_acc=0.977 test_f1=0.977


  Epoch 4 loss=0.0027 train_acc=0.999 val_acc=0.977 val_f1=0.977 test_acc=0.980 test_f1=0.980


  Epoch 5 loss=0.0004 train_acc=1.000 val_acc=0.973 val_f1=0.973 test_acc=0.987 test_f1=0.987
  Early stopping at epoch 5.

Evaluating HP combo: bs=4, lr=5e-06, wd=1e-05


KeyboardInterrupt: 

In [10]:
print("=" * 60)
print("10-FOLD CV — ENGLISH-TRAINED MODEL")
print("Train: ~90% English | Test: English fold + Full Tagalog + Balanced Combined")
print("=" * 60)

import pickle

# Update CHECKPOINT_DIR if not running on Colab
CHECKPOINT_DIR = "/content/checkpoints"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
CHECKPOINT_FILE = os.path.join(CHECKPOINT_DIR, "standard_cv_checkpoint.pkl")

N_SPLITS = 10

en_accs, en_f1s, en_precs, en_recs = [], [], [], []
tl_accs, tl_f1s, tl_precs, tl_recs = [], [], [], []
both_accs, both_f1s, both_precs, both_recs = [], [], [], []
en_f1_0, en_f1_1, en_rec_1, en_prec_1 = [], [], [], []
tl_f1_0, tl_f1_1, tl_rec_1, tl_prec_1 = [], [], [], []
both_f1_0, both_f1_1, both_rec_1, both_prec_1 = [], [], [], []

completed_seeds = set()
if os.path.exists(CHECKPOINT_FILE):
    print(f"Loading checkpoint from {CHECKPOINT_FILE}...")
    with open(CHECKPOINT_FILE, "rb") as f:
        checkpoint_data = pickle.load(f)
        en_accs = checkpoint_data.get("en_accs", [])
        en_f1s = checkpoint_data.get("en_f1s", [])
        en_precs = checkpoint_data.get("en_precs", [])
        en_recs = checkpoint_data.get("en_recs", [])
        tl_accs = checkpoint_data.get("tl_accs", [])
        tl_f1s = checkpoint_data.get("tl_f1s", [])
        tl_precs = checkpoint_data.get("tl_precs", [])
        tl_recs = checkpoint_data.get("tl_recs", [])
        both_accs = checkpoint_data.get("both_accs", [])
        both_f1s = checkpoint_data.get("both_f1s", [])
        both_precs = checkpoint_data.get("both_precs", [])
        both_recs = checkpoint_data.get("both_recs", [])
        en_f1_0 = checkpoint_data.get("en_f1_0", [])
        en_f1_1 = checkpoint_data.get("en_f1_1", [])
        en_rec_1 = checkpoint_data.get("en_rec_1", [])
        en_prec_1 = checkpoint_data.get("en_prec_1", [])
        tl_f1_0 = checkpoint_data.get("tl_f1_0", [])
        tl_f1_1 = checkpoint_data.get("tl_f1_1", [])
        tl_rec_1 = checkpoint_data.get("tl_rec_1", [])
        tl_prec_1 = checkpoint_data.get("tl_prec_1", [])
        both_f1_0 = checkpoint_data.get("both_f1_0", [])
        both_f1_1 = checkpoint_data.get("both_f1_1", [])
        both_rec_1 = checkpoint_data.get("both_rec_1", [])
        both_prec_1 = checkpoint_data.get("both_prec_1", [])
        completed_seeds = checkpoint_data.get("completed_seeds", set())
    seed_str = "seed" if len(completed_seeds) == 1 else "seeds"
    print(f"Loaded results for {len(completed_seeds)} {seed_str}: {completed_seeds}")

for seed in [s for s in SEEDS if s not in completed_seeds]:
    print(f"\n{'='*20} Seed: {seed} {'='*20}")
    set_seed(seed)
    clear_gpu()

    seed_en_accs, seed_en_f1s, seed_en_precs, seed_en_recs = [], [], [], []
    seed_tl_accs, seed_tl_f1s, seed_tl_precs, seed_tl_recs = [], [], [], []
    seed_both_accs, seed_both_f1s, seed_both_precs, seed_both_recs = [], [], [], []
    seed_en_f1_0, seed_en_f1_1, seed_en_rec_1, seed_en_prec_1 = [], [], [], []
    seed_tl_f1_0, seed_tl_f1_1, seed_tl_rec_1, seed_tl_prec_1 = [], [], [], []
    seed_both_f1_0, seed_both_f1_1, seed_both_rec_1, seed_both_prec_1 = [], [], [], []

    skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=seed)

    for fold, (train_idx, test_idx) in enumerate(skf.split(df_en["text"], df_en["label"])):
        print(f"\n-- Seed {seed} | Fold {fold+1}/{N_SPLITS} --")
        clear_gpu()
        fold_seed = seed + fold
        set_seed(fold_seed)

        train_df = df_en.iloc[train_idx].reset_index(drop=True)

        print(f"  Training with best HPs: bs={BEST_BS}, lr={BEST_LR:.0e}, wd={BEST_WD:.0e}")
        set_seed(fold_seed + 200)
        clear_gpu()

        train_loader = DataLoader(DementiaDataset(train_df, tokenizer), batch_size=BEST_BS, shuffle=True)

        model     = NeoBERTClassifier(MODEL_NAME).to(device)
        optimizer = make_optimizer(model, BEST_LR, BEST_WD)
        criterion = nn.CrossEntropyLoss()

        num_training_steps = len(train_loader) * MAX_EPOCHS
        num_warmup_steps   = int(0.1 * num_training_steps)
        scheduler = get_linear_schedule_with_warmup(
            optimizer,
            num_warmup_steps=num_warmup_steps,
            num_training_steps=num_training_steps
        )

        try:
            for epoch in range(MAX_EPOCHS):
                loss, train_acc = train_epoch(model, train_loader, optimizer, criterion, scheduler)
                if not np.isfinite(loss):
                    print(f"    Epoch {epoch+1}: NaN loss, stopping.")
                    break
                print(f"    Epoch {epoch+1} loss={loss:.4f} train_acc={train_acc:.3f}")
        finally:
            del optimizer, criterion, scheduler
            clear_gpu()

        print(f"  Evaluating Fold {fold+1}...")
        model.eval()

        en_test_this_fold = df_en.iloc[test_idx].reset_index(drop=True)
        tl_full_df        = df_tl.copy()

        en_test_0_len = len(en_test_this_fold[en_test_this_fold['label'] == 0])
        en_test_1_len = len(en_test_this_fold[en_test_this_fold['label'] == 1])
        tl_full_0_len = len(tl_full_df[tl_full_df['label'] == 0])
        tl_full_1_len = len(tl_full_df[tl_full_df['label'] == 1])

        n_combined_test_class_0 = min(en_test_0_len, tl_full_0_len)
        n_combined_test_class_1 = min(en_test_1_len, tl_full_1_len)

        english_part_for_combined_list = []
        if n_combined_test_class_0 > 0:
            english_part_for_combined_list.append(
                en_test_this_fold[en_test_this_fold['label'] == 0].sample(n=n_combined_test_class_0, random_state=fold_seed + 400)
            )
        if n_combined_test_class_1 > 0:
            english_part_for_combined_list.append(
                en_test_this_fold[en_test_this_fold['label'] == 1].sample(n=n_combined_test_class_1, random_state=fold_seed + 400)
            )
        english_part_for_combined = pd.concat(english_part_for_combined_list).sample(frac=1, random_state=fold_seed + 400).reset_index(drop=True) if english_part_for_combined_list else pd.DataFrame(columns=en_test_this_fold.columns)

        tagalog_part_for_combined_list = []
        if n_combined_test_class_0 > 0:
            tagalog_part_for_combined_list.append(
                tl_full_df[tl_full_df['label'] == 0].sample(n=n_combined_test_class_0, random_state=fold_seed + 401)
            )
        if n_combined_test_class_1 > 0:
            tagalog_part_for_combined_list.append(
                tl_full_df[tl_full_df['label'] == 1].sample(n=n_combined_test_class_1, random_state=fold_seed + 401)
            )
        tagalog_part_for_combined = pd.concat(tagalog_part_for_combined_list).sample(frac=1, random_state=fold_seed + 401).reset_index(drop=True) if tagalog_part_for_combined_list else pd.DataFrame(columns=tl_full_df.columns)

        test_both_final = pd.concat([english_part_for_combined, tagalog_part_for_combined]).reset_index(drop=True)

        test_en_loader   = DataLoader(DementiaDataset(en_test_this_fold, tokenizer), batch_size=BEST_BS)
        test_tl_loader   = DataLoader(DementiaDataset(tl_full_df,        tokenizer), batch_size=BEST_BS)
        test_both_loader = DataLoader(DementiaDataset(test_both_final,   tokenizer), batch_size=BEST_BS)

        acc, f1, prec, rec, f1_per, prec_per, rec_per = evaluate(model, test_en_loader, desc="    Test EN")
        print(f"      English  — Acc: {acc:.3f}, F1: {f1:.3f}")
        seed_en_accs.append(acc); seed_en_f1s.append(f1); seed_en_precs.append(prec); seed_en_recs.append(rec)
        seed_en_f1_0.append(f1_per[0]); seed_en_f1_1.append(f1_per[1]); seed_en_rec_1.append(rec_per[1]); seed_en_prec_1.append(prec_per[1])

        acc, f1, prec, rec, f1_per, prec_per, rec_per = evaluate(model, test_tl_loader, desc="    Test TL")
        print(f"      Tagalog  — Acc: {acc:.3f}, F1: {f1:.3f}")
        seed_tl_accs.append(acc); seed_tl_f1s.append(f1); seed_tl_precs.append(prec); seed_tl_recs.append(rec)
        seed_tl_f1_0.append(f1_per[0]); seed_tl_f1_1.append(f1_per[1]); seed_tl_rec_1.append(rec_per[1]); seed_tl_prec_1.append(prec_per[1])

        acc, f1, prec, rec, f1_per, prec_per, rec_per = evaluate(model, test_both_loader, desc="    Test Both")
        print(f"      Combined — Acc: {acc:.3f}, F1: {f1:.3f}")
        seed_both_accs.append(acc); seed_both_f1s.append(f1); seed_both_precs.append(prec); seed_both_recs.append(rec)
        seed_both_f1_0.append(f1_per[0]); seed_both_f1_1.append(f1_per[1]); both_rec_1.append(rec_per[1]); both_prec_1.append(seed_both_prec_1)

        del model, test_en_loader, test_tl_loader, test_both_loader
        clear_gpu()

    if seed_en_f1s:

        en_accs.extend(seed_en_accs);   en_f1s.extend(seed_en_f1s);   en_precs.extend(seed_en_precs);   en_recs.extend(seed_en_recs)
        en_f1_0.extend(seed_en_f1_0); en_f1_1.extend(seed_en_f1_1); en_rec_1.extend(seed_en_rec_1); en_prec_1.extend(seed_en_prec_1)
        tl_accs.extend(seed_tl_accs);   tl_f1s.extend(seed_tl_f1s);   tl_precs.extend(seed_tl_precs);   tl_recs.extend(seed_tl_recs)
        tl_f1_0.extend(seed_tl_f1_0); tl_f1_1.extend(seed_tl_f1_1); tl_rec_1.extend(seed_tl_rec_1); tl_prec_1.extend(seed_tl_prec_1)
        both_accs.extend(seed_both_accs); both_f1s.extend(seed_both_f1s); both_precs.extend(seed_both_precs); both_recs.extend(seed_both_recs)
        both_f1_0.extend(seed_both_f1_0); both_f1_1.extend(seed_both_f1_1); both_rec_1.extend(seed_both_rec_1); both_prec_1.extend(seed_both_prec_1)

        completed_seeds.add(seed)
        checkpoint_data = {
            "en_accs": en_accs, "en_f1s": en_f1s, "en_precs": en_precs, "en_recs": en_recs,
            "tl_accs": tl_accs, "tl_f1s": tl_f1s, "tl_precs": tl_precs, "tl_recs": tl_recs,
            "both_accs": both_accs, "both_f1s": both_f1s, "both_precs": both_precs, "both_recs": both_recs,
            "en_f1_0": en_f1_0, "en_f1_1": en_f1_1, "en_rec_1": en_rec_1, "en_prec_1": en_prec_1,
            "tl_f1_0": tl_f1_0, "tl_f1_1": tl_f1_1, "tl_rec_1": tl_rec_1, "tl_prec_1": tl_prec_1,
            "both_f1_0": both_f1_0, "both_f1_1": both_f1_1, "both_rec_1": both_rec_1, "both_prec_1": both_prec_1,
            "completed_seeds": completed_seeds
        }
        with open(CHECKPOINT_FILE, "wb") as f:
            pickle.dump(checkpoint_data, f)
        print(f"  Checkpoint saved for seed {seed}.")
    else:
        print(f"  No valid results for seed {seed}.")

    print(f"{'='*20} Finished Seed: {seed} {'='*20}")
    clear_gpu()

print("\nCV Complete.")

10-FOLD CV — ENGLISH-TRAINED MODEL
Train: ~90% English | Test: English fold + Full Tagalog + Balanced Combined

==================== Seed: 42 ====================

-- Seed 42 | Fold 1/10 --
  Training with best HPs: bs=4, lr=5e-06, wd=1e-02


KeyboardInterrupt: 

In [11]:
def fmt(vals):
    if len(vals) == 0:
        return "nan±nan"
    vals = np.asarray(vals, dtype=float)
    return f"{np.mean(vals):.3f}±{np.std(vals):.3f}"

print("\n" + "=" * 75)
print("ENGLISH-TRAINED NEOBERT — FINAL RESULTS (mean ± std across folds)")
print("=" * 75)

print(f"\n{'Metric':<28} {'English':>14} {'Tagalog':>14} {'Combined':>14}")
print("-" * 70)
print(f"{'Accuracy':<28} {fmt(en_accs):>14} {fmt(tl_accs):>14} {fmt(both_accs):>14}")
print(f"{'F1 Macro':<28} {fmt(en_f1s):>14} {fmt(tl_f1s):>14} {fmt(both_f1s):>14}")
print(f"{'Precision Macro':<28} {fmt(en_precs):>14} {fmt(tl_precs):>14} {fmt(both_precs):>14}")
print(f"{'Recall Macro':<28} {fmt(en_recs):>14} {fmt(tl_recs):>14} {fmt(both_recs):>14}")
print("-" * 70)
print(f"{'F1 Healthy (class 0)':<28} {fmt(en_f1_0):>14} {fmt(tl_f1_0):>14} {fmt(both_f1_0):>14}")
print(f"{'F1 AD (class 1)':<28} {fmt(en_f1_1):>14} {fmt(tl_f1_1):>14} {fmt(both_f1_1):>14}")
print(f"{'AD Recall (sensitivity)':<28} {fmt(en_rec_1):>14} {fmt(tl_rec_1):>14} {fmt(both_rec_1):>14}")
print(f"{'AD Precision (PPV)':<28} {fmt(en_prec_1):>14} {fmt(tl_prec_1):>14} {fmt(both_prec_1):>14}")
print("-" * 70)

if len(en_f1s) > 0 and len(tl_f1s) > 0:
    combined_f1_diffs = np.array(en_f1s) - np.array(tl_f1s)
    gap     = np.mean(combined_f1_diffs)
    gap_std = np.std(combined_f1_diffs)
    print(f"\nCross-lingual bias gap (EN - TL F1 macro): {gap:.3f}±{gap_std:.3f}")
else:
    print("\nCross-lingual bias gap (EN - TL F1 macro): nan±nan")


ENGLISH-TRAINED NEOBERT — FINAL RESULTS (mean ± std across folds)

Metric                              English        Tagalog       Combined
----------------------------------------------------------------------
Accuracy                            nan±nan        nan±nan        nan±nan
F1 Macro                            nan±nan        nan±nan        nan±nan
Precision Macro                     nan±nan        nan±nan        nan±nan
Recall Macro                        nan±nan        nan±nan        nan±nan
----------------------------------------------------------------------
F1 Healthy (class 0)                nan±nan        nan±nan        nan±nan
F1 AD (class 1)                     nan±nan        nan±nan        nan±nan
AD Recall (sensitivity)             nan±nan        nan±nan        nan±nan
AD Precision (PPV)                  nan±nan        nan±nan        nan±nan
----------------------------------------------------------------------

Cross-lingual bias gap (EN - TL F1 macro): nan±nan
